# Sprint 1 — ByteDance IaaS Cross-Validation

**What this does:** Patches `sprint1_Main.py` for ByteDance IaaS data (10-min intervals, CPU-only, different column names) and runs the full pipeline on Colab Pro+.

**Prerequisites:**
- `sprint1_Main.py` and `bytedance_runner.py` on Google Drive
- ByteDance data already split into `train.parquet`, `val.parquet`, `test.parquet`

**Expected runtime:** ~2–4 hours on A100, ~4–8 hours on T4 (93 containers × 4 horizons)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
import torch
print(f"PyTorch {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({props.total_memory/1e9:.1f} GB)")
else:
    print("WARNING: No GPU detected — BiLSTM will be very slow")


## Paths
Edit these if your Drive layout differs.


In [ ]:
import os

# ── EDIT THESE IF NEEDED ──────────────────────────────────────────────
DRIVE       = "/content/drive/MyDrive"
SPRINT_SRC  = f"{DRIVE}/sprint1 v9 results/sprint1_Main.py"
RUNNER      = f"{DRIVE}/sprint1 v9 results/bytedance_runner.py"
DATA_DIR    = f"{DRIVE}/workspace/thesis/bytedance/pipeline_data"
OUTPUT_DIR  = f"{DRIVE}/workspace/thesis/bytedance/results"
# ──────────────────────────────────────────────────────────────────────

# Verify everything exists
for label, path in [("sprint1_Main.py", SPRINT_SRC),
                    ("bytedance_runner.py", RUNNER),
                    ("data dir", DATA_DIR)]:
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}]  {label}: {path}")

# Check data splits
import pandas as pd
for split in ["train", "val", "test"]:
    p = os.path.join(DATA_DIR, f"{split}.parquet")
    if os.path.exists(p):
        df = pd.read_parquet(p)
        print(f"  {split}.parquet: {len(df):,} rows, {df.columns.tolist()[:5]}...")
    else:
        print(f"  {split}.parquet: MISSING")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nOutput dir: {OUTPUT_DIR}")


## Step 1: Copy scripts to local disk
Colab runs faster from local `/content` than from Drive.


In [ ]:
import shutil

shutil.copy(SPRINT_SRC, "/content/sprint1_Main.py")
shutil.copy(RUNNER, "/content/bytedance_runner.py")
print("Copied sprint1_Main.py and bytedance_runner.py to /content/")


## Step 2: Patch sprint1_Main.py → sprint1_bytedance.py
Applies all 11 ByteDance patches and verifies each one.


In [ ]:
!python3 /content/bytedance_runner.py --no-run --sprint-src /content/sprint1_Main.py


## Step 3: Verify the patched script


In [ ]:
# Quick sanity check — these should all print matching lines
!grep -n "SAMPLING_INTERVAL.*600" /content/sprint1_bytedance.py
!grep -n "ppd = 144" /content/sprint1_bytedance.py
!grep -n "SKIP_MEM_ENSEMBLE.*True" /content/sprint1_bytedance.py
!grep -n '"10min": 1' /content/sprint1_bytedance.py

# Syntax check
import py_compile
py_compile.compile('/content/sprint1_bytedance.py', doraise=True)
print("\nSyntax: OK")


## Step 4: Install missing dependencies (if any)


In [ ]:
!pip install optuna --quiet 2>/dev/null || true
# Most deps (xgboost, lightgbm, sklearn, torch) are pre-installed on Colab


## Step 5: Run the pipeline

**Flags explained:**
- `--sequential`: Run horizons one at a time (safer on Colab, avoids OOM)
- `--bilstm-batch`: BiLSTM batch size — adjust by GPU memory
- `--aci-buffer 500`: Smaller buffer for 93 containers (vs 2000 for Alibaba's 4919)

| GPU | `--bilstm-batch` | Estimated time |
|-----|-------------------|----------------|
| A100 (40 GB) | 2048 | ~2–3 hours |
| L4 (24 GB) | 1024 | ~3–5 hours |
| T4 (16 GB) | 512 | ~4–8 hours |

**Checkpointing:** The pipeline saves after each stage. If Colab disconnects, re-run this cell — it skips completed stages.


In [ ]:
# Auto-detect batch size based on GPU memory
import torch
if torch.cuda.is_available():
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if mem_gb >= 35:
        batch = 2048   # A100
    elif mem_gb >= 20:
        batch = 1024   # L4
    else:
        batch = 512    # T4
    print(f"GPU: {torch.cuda.get_device_name(0)} ({mem_gb:.0f} GB) → batch={batch}")
else:
    batch = 256
    print("No GPU → batch=256 (will be slow)")

import subprocess, sys
cmd = [
    sys.executable, "/content/sprint1_bytedance.py",
    "--data-dir",   DATA_DIR,
    "--output-dir",  OUTPUT_DIR,
    "--sequential",
    "--bilstm-batch", str(batch),
    "--aci-buffer",  "500",
]
print(f"Running: {' '.join(cmd)}\n")
subprocess.run(cmd)


## Step 6: View results


In [ ]:
import pandas as pd
import os

results_path = os.path.join(OUTPUT_DIR, "results_summary.csv")
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    cols = ["Horizon", "naive_R2", "hetero_ensemble_R2", "cpu_improvement_pp"]
    available = [c for c in cols if c in df.columns]
    print(df[available].to_string(index=False))
    print()
    print("Full results saved to:", results_path)
else:
    print("results_summary.csv not found yet")
    # Check per-horizon progress
    import json
    for hz in ["10min", "30min", "60min", "120min"]:
        state_path = os.path.join(OUTPUT_DIR, hz, "_state.json")
        if os.path.exists(state_path):
            with open(state_path) as f:
                state = json.load(f)
            done = [k for k, v in state.get("done", {}).items() if v]
            print(f"  {hz}: {len(done)} stages complete")


## Step 7: Verify checkpoints on Drive
All results are on Drive already (output-dir points there). This just confirms.


In [ ]:
import os

print("Checkpoint summary:")
for hz in ["10min", "30min", "60min", "120min"]:
    hz_dir = os.path.join(OUTPUT_DIR, hz)
    if os.path.exists(hz_dir):
        files = os.listdir(hz_dir)
        size_mb = sum(os.path.getsize(os.path.join(hz_dir, f)) for f in files) / 1e6
        print(f"  {hz}: {len(files)} files, {size_mb:.1f} MB")
    else:
        print(f"  {hz}: not started")


## Step 8: Compare with previous run (optional)
If you had a previous ByteDance run, compare here.


In [ ]:
import pandas as pd
import os

results_path = os.path.join(OUTPUT_DIR, "results_summary.csv")
if os.path.exists(results_path):
    df = pd.read_csv(results_path)

    print("ByteDance IaaS Results")
    print("=" * 60)
    for _, row in df.iterrows():
        hz = row.get("Horizon", "?")
        naive = row.get("naive_R2", 0)
        ens = row.get("hetero_ensemble_R2", 0)
        pp = row.get("cpu_improvement_pp", 0)
        print(f"  {hz:>6}  naive R2={naive:.4f}  ensemble R2={ens:.4f}  +{pp:.2f}pp")

    print()
    print("Alibaba comparison (from sprint1):")
    print("  10min  naive R2=0.9188  ensemble R2=0.9214  +0.25pp")
    print("  30min  naive R2=0.8361  ensemble R2=0.8404  +0.43pp")
    print("  60min  naive R2=0.7878  ensemble R2=0.8011  +1.33pp")
    print("  120min naive R2=0.7178  ensemble R2=0.7642  +4.64pp")
